In [6]:
!pip install -e ../. 

Obtaining file:///Users/fabian/Python/REANIMATOR%20SIGIR/Reanimator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for reanimator (pyproject.toml) ... done
  Created wheel for reanimator: filename=reanimator-0.1.4-0.editable-py3-none-any.whl size=5586 sha256=70bfb7224c512161d1170319a97e2541cfaae7bbf591666372fc067fa4637722
  Stored in directory: /private/var/folders/n_/ndw79bp52bx29ynt1q6l44rw0000gn/T/pip-ephem-wheel-cache-jwwdyc9m/wheels/d2/3d/11/4f5c8951501fef9f5bf6fa1404717b91741fc20e3ea1e6b7d7
Successfully built reanimator
  Attempting uninstall: reanimator
    Found existing installation: reanimator 0.1.4
    Uninstalling reanimator-0.1.4:
      Successfully uninstalled reanimator-0.1.4

[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler
from reanimator.labelers import TopicChunkPair, calculate_cohens_kappa
load_dotenv()

import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/fabian/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/src/reanimator/sources.py:18: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [3]:
human_judgements = reanimator.source.get_qrels()
topics = reanimator.source.get_topics()
model="qwen/qwen3-30b-a3b"

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.


In [4]:
t1_doc_ids = [judg.doc_id for judg in human_judgements if judg.query_id == "1"]
len(t1_doc_ids)

1647

In [5]:
docs = reanimator.load_documents(doc_ids=t1_doc_ids)[:10]


Step 1: Loading documents from source...


cord19/trec-covid documents: 100%|██████████| 192509/192509 [00:01<00:00, 172446.58it/s]


In [6]:
reanimator.download_documents(docs)
reanimator.extract_content(docs)
reanimator.save_documents(docs, "data/documents")


Step 2: Fetching URLs and downloading PDFs...
All DOIs already have cached URLs.


Failed to download 10.1093/nar/gkq1013: 403 Client Error: Forbidden for url: https://academic.oup.com/nar/article-pdf/39/suppl_1/D569/18784692/gkq1013.pdf
Failed to download 10.1093/nar/gkq089: 403 Client Error: Forbidden for url: https://academic.oup.com/nar/article-pdf/38/9/e111/33236399/gkq089.pdf
Failed to download 10.1155/2011/284795: 403 Client Error: Forbidden for url: https://downloads.hindawi.com/journals/ecam/2011/284795.pdf
PDF downloading complete.

Step 3: Extracting content from PDFs...


Extracting Content:   0%|          | 0/10 [00:00<?, ?it/s]/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory'

Saving 10 documents to directory documents...
Finished saving documents.


In [7]:
chunks = reanimator.chunker.chunk(docs, metadata_fields_to_chunk=["title"])

In [8]:
docs = reanimator.load_documents_from_file("data/documents")

Attempting to load documents from 10 files...
Successfully loaded 10 documents.


In [9]:
labeler = OpenAILabeler(api_key=os.getenv("OPENAI_API_KEY"), prompt_path="../src/reanimator/default_prompt.txt")
model = labeler.model

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


In [ ]:
labeler = LocalModelLabeler(model="qwen/qwen3-30b-a3b", 
                            base_url="http://192.168.178.180:1234/v1", 
                            concurrency=10,
                            thinking=False)

INFO: LocalModelLabeler initialized with model: qwen/qwen3-30b-a3b at http://192.168.178.180:1234/v1


In [10]:
batch = [TopicChunkPair(topic=topics[0], chunk=chunk) for chunk in chunks]

In [11]:
machine_judgements = await labeler.label_all(batch)

Generating Judgements: 100%|██████████| 1057/1057 [01:27<00:00, 12.10it/s]


In [12]:
from reanimator.models import save_judgements
model = model.replace("/", "_")
save_judgements(machine_judgements, f"machine_{model}_judgements.json")

## Human Relevance Judgments


In [13]:
from reanimator.human_labeling import *

In [14]:
# Load machine labeled pairs here:
with open('machine_gpt-4.1-mini-2025-04-14_judgements.json') as f:
    machine_judgements = json.load(f)

In [15]:
# insert path to your labeling log file 
output_path = "fabian_judgements.json"
# what modality are you labeling: text or table
modality = "text"

In [16]:
label_chunks = [a.to_dict() for a in chunks]
label_topics = [t.to_dict() for t in topics]

In [17]:
to_label = load_label_pairs(machine_judgements, label_chunks, output_path="fabian_judgements.json", modality=modality, num_pairs=20)

17  texts left to label: 20 pairs to label, 3 already labeled in 'fabian_judgements.json'.


In [18]:
labeling_interface(output_path="fabian_judgements.json", sampled=to_label, topics=label_topics, chunks=label_chunks)

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

...

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

In [ ]:
calculate_cohens_kappa("/workspace/data/judgments/human_judgements.json", "/workspace/data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [7]:
from reanimator.retrieval import Indexer, Retriever

indexer = Indexer(index_type="bm25", bm25_path="/workspace/data/indices/bm25_retriever.pkl", max_docs=200)

In [8]:
indexer.index(chunks)

Creating new indexes...
Indexes created and saved.


In [9]:
retriever = Retriever(indexer)

In [10]:
retriever.retrieve(topics[0].query_text)

{'sparse_coronavirus origin': Ranking(query_id='ad-hoc', results=[SearchResult(doc_id='uadfehr6', score=0, rank=0, chunk_id='uadfehr6-metadata-title-0', metadata={}), SearchResult(doc_id='73xil5op', score=0, rank=1, chunk_id='73xil5op-metadata-title-0', metadata={}), SearchResult(doc_id='o877uul1', score=0, rank=2, chunk_id='o877uul1-metadata-title-0', metadata={}), SearchResult(doc_id='w53u5ive', score=0, rank=3, chunk_id='w53u5ive-metadata-title-0', metadata={}), SearchResult(doc_id='es7q6c90', score=0, rank=4, chunk_id='es7q6c90-metadata-title-0', metadata={}), SearchResult(doc_id='hncf2qe8', score=0, rank=5, chunk_id='hncf2qe8-metadata-title-0', metadata={}), SearchResult(doc_id='bgialj4d', score=0, rank=6, chunk_id='bgialj4d-metadata-title-0', metadata={}), SearchResult(doc_id='1ag9jkk6', score=0, rank=7, chunk_id='1ag9jkk6-metadata-title-0', metadata={}), SearchResult(doc_id='beguhous', score=0, rank=8, chunk_id='beguhous-metadata-title-0', metadata={}), SearchResult(doc_id='jkej